In [2]:
#importing the required libraries
import os
import glob
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [3]:
# uploading the folder containing the data files
cancer_folder = '/Users/sunilverma/Downloads/Dataset/Cancer'
non_cancer_folder = '/Users/sunilverma/Downloads/Dataset/Non_Cancer'
  # replace with your folder path

In [4]:
import os
import glob
import pandas as pd

def load_folder_to_df(folder_path, label):
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))  # scan for .txt files
    print(f"Scanning {folder_path}, found {len(all_files)} files")  # Debug line
    df_list = []

    for file in all_files:
        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read().strip()

        # Initialize defaults
        abstract_id, title, abstract = None, None, None

        # Parse line by line
        for line in content.splitlines():
            line = line.strip()
            if line.startswith("<ID:"):
                abstract_id = line.replace("<ID:", "").replace(">", "").strip()
            elif line.startswith("Title:"):
                title = line.replace("Title:", "").strip()
            elif line.startswith("Abstract:"):
                abstract = line.replace("Abstract:", "").strip()
            else:
                # Append continuation lines to abstract (if multi-line)
                if abstract is not None:
                    abstract += " " + line

        df_list.append({
            "abstract_id": abstract_id if abstract_id else os.path.basename(file),
            "title": title,
            "abstract": abstract,
            "label": label
        })

    return pd.DataFrame(df_list)


# ✅ Usage example (Colab paths)
# cancer_folder = "/content/drive/MyDrive/Dataset/Cancer"
# non_cancer_folder = "/content/drive/MyDrive/Dataset/Non-Cancer"

cancer_df = load_folder_to_df(cancer_folder, "Cancer")
non_cancer_df = load_folder_to_df(non_cancer_folder, "Non-Cancer")

combined_df = pd.concat([cancer_df, non_cancer_df], ignore_index=True)

print(combined_df.head())
print("Total rows:", combined_df.shape)



Scanning /Users/sunilverma/Downloads/Dataset/Cancer, found 500 files
Scanning /Users/sunilverma/Downloads/Dataset/Non_Cancer, found 0 files
  abstract_id                                              title  \
0    31055803  [Analysis of age-specific cytogenetic changes ...   
1    31164412  T-Cell Deletion of MyD88 Connects IL17 and Ika...   
2    31094905  MYCN Amplified Relapse Following Resolution of...   
3    31498304  In Vivo Inhibition of MicroRNA to Decrease Tum...   
4    30897768  Breast Cancer and miR-SNPs: The Importance of ...   

                                            abstract   label  
0  OBJECTIVE: To characterize cytogenetic changes...  Cancer  
1  Cancer development requires a favorable tissue...  Cancer  
2  Congenital neuroblastoma with placental involv...  Cancer  
3  MicroRNAs (miRNAs) are important regulators of...  Cancer  
4  Recent studies in cancer diagnostics have iden...  Cancer  
Total rows: (500, 4)


In [5]:
combined_df.head(20)


,abstract_id,title,abstract,label
0,31055803,[Analysis of age-specific cytogenetic changes ...,OBJECTIVE: To characterize cytogenetic changes...,Cancer
1,31164412,T-Cell Deletion of MyD88 Connects IL17 and Ika...,Cancer development requires a favorable tissue...,Cancer
2,31094905,MYCN Amplified Relapse Following Resolution of...,Congenital neuroblastoma with placental involv...,Cancer
3,31498304,In Vivo Inhibition of MicroRNA to Decrease Tum...,MicroRNAs (miRNAs) are important regulators of...,Cancer
4,30897768,Breast Cancer and miR-SNPs: The Importance of ...,Recent studies in cancer diagnostics have iden...,Cancer
5,31539535,Estrogen signaling and estrogen receptors as p...,Laryngeal squamous cell carcinoma (LSCC) has b...,Cancer
6,36791667,A global collaboRAtive study of CIC-rearranged...,Undifferentiated small round cell sarcomas (UR...,Cancer
7,31027701,Improvement in the survival of patients with s...,OBJECTIVES: In the past two decades several an...,Cancer
8,35608106,Targeting <i>FGFR3</i> alterations with adjuva...,"PROOF 302 is an ongoing randomized, double-bli...",Cancer
9,30989185,Isolated Bone Marrow Non-Langerhans Cell Histi...,OBJECTIVES: The prevalence of concomitant myel...,Cancer


# Task
Analyze and classify research paper abstracts from "combined_df" into cancer and non-cancer categories, extract specific diseases mentioned in each abstract, and evaluate the performance of a fine-tuned language model compared to a baseline model using confusion matrices.

## Data preprocessing

### Subtask:
Clean and prepare the `combined_df` DataFrame by removing redundant metadata, normalizing citations, and handling missing abstracts.


**Reasoning**:
Inspect the combined_df DataFrame for redundant metadata and patterns, check for missing values in the 'abstract' column, and handle them by dropping rows with missing abstracts.



In [6]:
print("Initial shape of combined_df:", combined_df.shape)
print("Missing values in 'abstract' column before handling:", combined_df['abstract'].isnull().sum())

# Handle missing abstracts by dropping rows
combined_df.dropna(subset=['abstract'], inplace=True)

print("Shape of combined_df after handling missing abstracts:", combined_df.shape)
print("Missing values in 'abstract' column after handling:", combined_df['abstract'].isnull().sum())

# Display the first few rows to inspect for redundant metadata or patterns
display(combined_df.head())

Initial shape of combined_df: (500, 4)
Missing values in 'abstract' column before handling: 0
Shape of combined_df after handling missing abstracts: (500, 4)
Missing values in 'abstract' column after handling: 0


,abstract_id,title,abstract,label
0,31055803,[Analysis of age-specific cytogenetic changes ...,OBJECTIVE: To characterize cytogenetic changes...,Cancer
1,31164412,T-Cell Deletion of MyD88 Connects IL17 and Ika...,Cancer development requires a favorable tissue...,Cancer
2,31094905,MYCN Amplified Relapse Following Resolution of...,Congenital neuroblastoma with placental involv...,Cancer
3,31498304,In Vivo Inhibition of MicroRNA to Decrease Tum...,MicroRNAs (miRNAs) are important regulators of...,Cancer
4,30897768,Breast Cancer and miR-SNPs: The Importance of ...,Recent studies in cancer diagnostics have iden...,Cancer


**Reasoning**:
Normalize citations in the 'title' and 'abstract' columns using regular expressions and then display the first few rows and the shape of the processed DataFrame.



In [7]:
import re

def normalize_citations(text):
    # This regex pattern attempts to find common citation patterns like [1], [2,3], [4-6], or (Smith, 2020)
    # It's a simplification and may not catch all patterns, but covers common cases.
    citation_pattern = r'\[\d+(?:,\s*\d+)*\]|\(\w+,\s*\d{4}\)'
    if isinstance(text, str):
        # Replace found citation patterns with an empty string
        cleaned_text = re.sub(citation_pattern, '', text)
        return cleaned_text.strip()
    return text # Return original if not a string

# Apply the normalization function to 'title' and 'abstract' columns
combined_df['title'] = combined_df['title'].apply(normalize_citations)
combined_df['abstract'] = combined_df['abstract'].apply(normalize_citations)

# Display the first few rows and the shape of the processed DataFrame
display(combined_df.head())
print("Shape of combined_df after cleaning:", combined_df.shape)

,abstract_id,title,abstract,label
0,31055803,[Analysis of age-specific cytogenetic changes ...,OBJECTIVE: To characterize cytogenetic changes...,Cancer
1,31164412,T-Cell Deletion of MyD88 Connects IL17 and Ika...,Cancer development requires a favorable tissue...,Cancer
2,31094905,MYCN Amplified Relapse Following Resolution of...,Congenital neuroblastoma with placental involv...,Cancer
3,31498304,In Vivo Inhibition of MicroRNA to Decrease Tum...,MicroRNAs (miRNAs) are important regulators of...,Cancer
4,30897768,Breast Cancer and miR-SNPs: The Importance of ...,Recent studies in cancer diagnostics have iden...,Cancer


Shape of combined_df after cleaning: (500, 4)


## Model selection

### Subtask:
Choose a suitable pre-trained multi-label text classification model (e.g., from Hugging Face Transformers) and justify the selection.


**Reasoning**:
Select a suitable pre-trained transformer model from Hugging Face for text classification and provide a justification.



In [8]:
# Model Selection Justification

# For text classification tasks, especially on abstracts which are relatively short to medium length texts,
# models like BERT and RoBERTa are strong candidates. BERT (Bidirectional Encoder Representations from Transformers)
# is a foundational model that has shown excellent performance on a wide range of NLP tasks.
# RoBERTa (A Robustly Optimized BERT Pretraining Approach) is a re-trained version of BERT
# with improved training methodology, often leading to better performance.

# Considering the task is binary classification (Cancer vs Non-Cancer) and the input are scientific abstracts,
# a well-established and widely used model like 'bert-base-uncased' or 'roberta-base' would be suitable.
# These base models offer a good balance between performance and computational requirements compared to
# larger models.

# Justification for choosing 'bert-base-uncased':
# 1. Widely used and well-documented: 'bert-base-uncased' is one of the most popular pre-trained models,
#    making it easy to find resources, examples, and community support.
# 2. Suitable for text classification: BERT's architecture is designed to understand the context of words
#    in a bidirectional manner, which is crucial for capturing the nuances in scientific text.
# 3. 'uncased' version: The 'uncased' version means it does not distinguish between uppercase and lowercase
#    letters, which is generally acceptable for text classification where the meaning is less dependent on case.
# 4. Base model size: The 'base' model has a reasonable number of parameters (110M), making it feasible
#    to fine-tune on standard hardware (like a Colab GPU) without excessive memory or computation time.

# While RoBERTa might offer slightly better performance, BERT is a solid and reliable choice for a baseline
# fine-tuned model in this scenario, especially for demonstrating the process and comparing against a simple
# baseline model later.

# Selected model identifier from Hugging Face:
model_name = 'bert-base-uncased'

print(f"Selected model: {model_name}")

Selected model: bert-base-uncased


## Data preparation for model training

### Subtask:
Split the preprocessed data into training, validation, and test sets. Tokenize the text data for the chosen model.


**Reasoning**:
Import necessary libraries for splitting data and tokenization, define features and labels, and split the data into training, validation, and test sets with stratification.



In [9]:
#! pip install transformers

In [10]:
#! pip install --upgrade transformers==4.30.0

In [11]:
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast

# Define features (abstract text) and labels (cancer/non-cancer)
X = combined_df['abstract']
y = combined_df['label']

# Split data into training (70%) and temp (30%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Split temp data into validation (15%) and test (15%) sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print(f"Training label distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Validation label distribution:\n{y_val.value_counts(normalize=True)}")
print(f"Test label distribution:\n{y_test.value_counts(normalize=True)}")

Training set size: 350
Validation set size: 75
Test set size: 75
Training label distribution:
label
Cancer    1.0
Name: proportion, dtype: float64
Validation label distribution:
label
Cancer    1.0
Name: proportion, dtype: float64
Test label distribution:
label
Cancer    1.0
Name: proportion, dtype: float64


**Reasoning**:
Instantiate the tokenizer, tokenize the text data for each set, and encode the labels into numerical format.



In [12]:
#! pip install torch

In [13]:
# Instantiate a BertTokenizerFast object
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# Tokenize the text data for each set
X_train_encoded = tokenizer(list(X_train), truncation=True, padding=True, return_tensors="pt")
X_val_encoded = tokenizer(list(X_val), truncation=True, padding=True, return_tensors="pt")
X_test_encoded = tokenizer(list(X_test), truncation=True, padding=True, return_tensors="pt")

# Encode the labels into numerical format
label_map = {'Non-Cancer': 0, 'Cancer': 1}
y_train_encoded = y_train.map(label_map)
y_val_encoded = y_val.map(label_map)
y_test_encoded = y_test.map(label_map)

print("X_train_encoded keys:", X_train_encoded.keys())
print("y_train_encoded sample:", y_train_encoded.head())

X_train_encoded keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])
y_train_encoded sample: 175    1
42     1
475    1
491    1
86     1
Name: label, dtype: int64


## Baseline model training and evaluation

### Subtask:
Train the selected model on the training data without fine-tuning and evaluate its performance using a confusion matrix.


**Reasoning**:
Train the selected model on the training data without fine-tuning and evaluate its performance using a confusion matrix and classification report.



In [18]:
! pip install accelerate -U

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np

# 0. Device handling (CPU vs MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU).")
else:
    device = torch.device("cpu")
    print("Using CPU.")

# 1. Define the number of labels
num_labels = len(y_train_encoded.unique())
print(f"Number of labels: {num_labels}")

# 2. Load the pre-trained bert-base-uncased model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
model.to(device)   # <- Move model to device
print("BERT model loaded.")

# 3. Define the TrainingArguments for the baseline model
training_args = TrainingArguments(
    output_dir="./baseline_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # 1 epoch = baseline
    logging_dir="./baseline_logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-8,   # tiny learning rate to avoid meaningful updates
    no_cuda=True          # <- prevents Trainer from forcing CUDA; allows CPU/MPS
)
print("Training arguments defined.")

# 4. Create a custom PyTorch Dataset class
class AbstractDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # force inputs to long
        item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
        # labels must also be long (class ids)
        item['labels'] = torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# 5. Instantiate Dataset objects
train_dataset = AbstractDataset(X_train_encoded, y_train_encoded)
val_dataset = AbstractDataset(X_val_encoded, y_val_encoded)
test_dataset = AbstractDataset(X_test_encoded, y_test_encoded)
print("Datasets instantiated.")

# 6. Instantiate a Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset # Evaluate on validation set during "training"
)
print("Trainer instantiated.")

# 7. Train the baseline model (minimal training as defined in args)
print("Starting baseline model training (minimal)...")
trainer.train()
print("Baseline model training finished.")

# 8. Make predictions on the test set
print("Making predictions on the test set...")
predictions = trainer.predict(test_dataset)
print("Predictions made.")

# 9. Convert predicted logits to class predictions
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids
print("Predicted labels:", predicted_labels[:10])
print("True labels:", true_labels[:10])

# 10. Compute and Display the Confusion Matrix
print("Computing and displaying confusion matrix...")
cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Cancer', 'Cancer'])
disp.plot()
print("Confusion matrix displayed.")

# 11. Print key classification metrics
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=['Non-Cancer', 'Cancer']))


Using MPS (Apple Silicon GPU).
Number of labels: 1


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

BERT model loaded.
Training arguments defined.
Datasets instantiated.
Trainer instantiated.
Starting baseline model training (minimal)...


RuntimeError: Found dtype Long but expected Float

In [14]:
# import torch
# from torch.utils.data import Dataset
# from transformers import BertForSequenceClassification, Trainer, TrainingArguments
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
# import numpy as np

# # 1. Define the number of labels
# num_labels = len(y_train_encoded.unique())
# print(f"Number of labels: {num_labels}")

# # 2. Load the pre-trained bert-base-uncased model
# model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
# print("BERT model loaded.")

# # 3. Define the TrainingArguments for the baseline model (no fine-tuning)
# # Setting a very small number of epochs and a large learning rate to simulate 'no fine-tuning'
# # Effectively, we are evaluating the pre-trained model's performance without significant training.
# # In a true 'baseline without fine-tuning' scenario, you might skip the Trainer and just use the model
# # directly for predictions. However, using Trainer allows for a consistent evaluation framework.
# # We'll set epochs to 1 and a minimal learning step to effectively use the pre-trained weights.
# training_args = TrainingArguments(
#     output_dir="./baseline_results",
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=1, # Very few epochs to simulate no significant training
#     logging_dir="./baseline_logs",
#     logging_steps=10,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=0.0 # No learning rate to prevent updates
# )
# print("Training arguments defined.")

# # 4. Create a custom PyTorch Dataset class
# class AbstractDataset(Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item['labels'] = torch.tensor(self.labels.iloc[idx])
#         return item

#     def __len__(self):
#         return len(self.labels)

# # 5. Instantiate Dataset objects
# train_dataset = AbstractDataset(X_train_encoded, y_train_encoded)
# val_dataset = AbstractDataset(X_val_encoded, y_val_encoded)
# test_dataset = AbstractDataset(X_test_encoded, y_test_encoded)
# print("Datasets instantiated.")

# # 6. Instantiate a Trainer object
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset # Evaluate on validation set during "training"
# )
# print("Trainer instantiated.")

# # 7. Train the baseline model (minimal training as defined in args)
# print("Starting baseline model training (minimal)...")
# trainer.train()
# print("Baseline model training finished.")

# # 8. Make predictions on the test set
# print("Making predictions on the test set...")
# predictions = trainer.predict(test_dataset)
# print("Predictions made.")

# # 9. Convert predicted logits to class predictions
# predicted_labels = np.argmax(predictions.predictions, axis=1)
# true_labels = predictions.label_ids
# print("Predicted labels:", predicted_labels[:10])
# print("True labels:", true_labels[:10])


# # 10. Compute and Display the Confusion Matrix
# print("Computing and displaying confusion matrix...")
# cm = confusion_matrix(true_labels, predicted_labels)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Cancer', 'Cancer'])
# disp.plot()
# print("Confusion matrix displayed.")

# # 11. Print key classification metrics
# print("\nClassification Report:")
# print(classification_report(true_labels, predicted_labels, target_names=['Non-Cancer', 'Cancer']))

Number of labels: 1


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

BERT model loaded.
Training arguments defined.
Datasets instantiated.
Trainer instantiated.
Starting baseline model training (minimal)...


RuntimeError: Placeholder storage has not been allocated on MPS device!

**Reasoning**:
The previous attempt failed because the `TrainingArguments` class initialization in the installed `transformers` library version does not accept the `evaluation_strategy` argument directly in the constructor. I need to fix the `TrainingArguments` instantiation to use the correct parameter names for evaluation and saving strategies, which were updated in newer versions of the library to `eval_strategy` and `save_strategy` or similar. I will regenerate the full code block with the corrected parameters.



In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np

# 1. Define the number of labels
num_labels = len(y_train_encoded.unique())
print(f"Number of labels: {num_labels}")

# 2. Load the pre-trained bert-base-uncased model
# Set ignore_mismatched_sizes=True if you are loading weights from a different head,
# but here we are loading the base model and adding a classification head, so it's not needed.
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
print("BERT model loaded.")

# 3. Define the TrainingArguments for the baseline model (no fine-tuning)
# Setting a very small number of epochs and a large learning rate to simulate 'no fine-tuning'
# Effectively, we are evaluating the pre-trained model's performance without significant training.
# In a true 'baseline without fine-tuning' scenario, you might skip the Trainer and just use the model
# directly for predictions. However, using Trainer allows for a consistent evaluation framework.
# We'll set epochs to 1 and a minimal learning step to effectively use the pre-trained weights.
# Correcting the parameter names based on potential version differences in transformers library.
training_args = TrainingArguments(
    output_dir="./baseline_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1, # Very few epochs to simulate no significant training
    logging_dir="./baseline_logs",
    logging_steps=10,
    # Use 'evaluation_strategy' if available, otherwise try 'eval_strategy' or specify 'Interval'
    # Let's try 'evaluation_strategy' again first, if it fails, try 'eval_strategy' or enum value
    evaluation_strategy="epoch", # Changed from evaluation_strategy to eval_strategy in some versions
    save_strategy="epoch", # Changed from save_strategy to save_strategy in some versions
    learning_rate=0.0 # No learning rate to prevent updates
)
print("Training arguments defined.")

# 4. Create a custom PyTorch Dataset class
class AbstractDataset(Dataset):
    def __init__(self, encodings, labels):
        # Ensure all tensors are on the same device if applicable, though Trainer handles device placement
        self.encodings = {key: val for key, val in encodings.items()} # No need to convert to tensor here, Trainer does it
        self.labels = labels # Store labels directly

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in self.encodings} # Access tensors directly
        item['labels'] = torch.tensor(self.labels.iloc[idx]) # Convert label to tensor
        return item

    def __len__(self):
        return len(self.labels)

# 5. Instantiate Dataset objects
train_dataset = AbstractDataset(X_train_encoded, y_train_encoded)
val_dataset = AbstractDataset(X_val_encoded, y_val_encoded)
test_dataset = AbstractDataset(X_test_encoded, y_test_encoded)
print("Datasets instantiated.")

# 6. Instantiate a Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset # Evaluate on validation set during "training"
)
print("Trainer instantiated.")

# 7. Train the baseline model (minimal training as defined in args)
print("Starting baseline model training (minimal)...")
trainer.train()
print("Baseline model training finished.")

# 8. Make predictions on the test set
print("Making predictions on the test set...")
predictions = trainer.predict(test_dataset)
print("Predictions made.")

# 9. Convert predicted logits to class predictions
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids
print("Predicted labels:", predicted_labels[:10])
print("True labels:", true_labels[:10])


# 10. Compute and Display the Confusion Matrix
print("Computing and displaying confusion matrix...")
cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Cancer', 'Cancer'])
disp.plot()
print("Confusion matrix displayed.")

# 11. Print key classification metrics
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=['Non-Cancer', 'Cancer']))

Number of labels: 2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT model loaded.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'